# EXP-009: Robust Per-Well Polynomial Projection Postprocess

**ðŸ–¥ RUNS ON: Kaggle CPU â€” no GPU needed**

| Item | Value |
|------|-------|
| Target env | Kaggle CPU (no accelerator) |
| Compute | ~15 min |
| Expected gain | ~0.3â€“0.5 RMSE (top-score notebook, CV-validated) |
| Risk | Low â€” pure postprocess, no retraining |

**Run this AFTER EXP-007** â€” applies per-well polynomial projection to EXP-007 OOF.

## What this does
Reproduced from `rogii-ridge-sp45-proj.ipynb` (LB 7.748):
- `U = TVT_pred_residual + dz`  where `dz = Z_eval - Z_last_known` (from train_df)
- `s = md_since / max(md_since)`  â€” normalized along-lateral position âˆˆ [0,1]
- Robust degree-5 polynomial fit via Tukey iterative reweighting (4 iters, c=2.0)
- `corrected_residual = U_fit - dz`

## IO
- **Input**: `PREP_DIR/train_df.joblib` (from rogii-eda-features.ipynb)
- **Input**: `WORK_DIR/EXP-007_final_oof.npy` (residual OOF from EXP-007)
- **Input** (test): raw `DATA_DIR/test/` horizontal well CSVs for MD/Z anchors
- **Output**: `WORK_DIR/EXP-009_oof.npy` + `EXP-009_test.npy` (residual)

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import joblib
from pathlib import Path
from sklearn.metrics import mean_squared_error

# â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
EXP_ID      = 'EXP-009'
TINY_SAMPLE = False
PROJ_DEG    = 5
PROJ_ITERS  = 4
PROJ_C      = 2.0

# â”€â”€ Paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
IS_KAGGLE = Path('/kaggle/input').exists()

# Input: output of rogii-eda-features.ipynb
PREP_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv05/rogii-eda-features'),
    Path('./output_aw'),
]
PREP_DIR = next((p for p in PREP_CANDIDATES if (p / 'train_df.joblib').exists()),
                PREP_CANDIDATES[-1])

# Input: prior OOF predictions (from exp007_ridge_stack__GPU.ipynb)
PRIOR_OOF_CANDIDATES = [
    Path('/kaggle/input/notebooks/baopv05/rogii-ridge-stack'),
    Path('./output_exp007'),
]
PRIOR_OOF_DIR = next((p for p in PRIOR_OOF_CANDIDATES if (p / 'EXP-007_final_oof.npy').exists() or (p / 'aw-blend_oof.npy').exists()),
                     PRIOR_OOF_CANDIDATES[-1])

# Input: raw competition data (for test projection anchors)
DATA_CANDIDATES = [
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('./rogii-wellbore-geology-prediction'),
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# Output
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('./output_exp009')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'EXP_ID      = {EXP_ID}')
print(f'TINY_SAMPLE = {TINY_SAMPLE}')
print(f'Projection: deg={PROJ_DEG}, iters={PROJ_ITERS}, c={PROJ_C}')
print(f'PREP_DIR    = {PREP_DIR}')
print(f'PRIOR_OOF_DIR = {PRIOR_OOF_DIR}')
print(f'DATA_DIR    = {DATA_DIR}')

In [ ]:
# â”€â”€ Projection postprocess functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _robfit(s, u, deg=5, n_iter=4, c=2.0):
    """Robust polynomial fit via Tukey iterative reweighting."""
    w = np.ones(len(s))
    u_fit = u.copy()
    for _ in range(n_iter):
        coeffs = np.polyfit(s, u, deg=deg, w=w)
        u_fit  = np.polyval(coeffs, s)
        resid  = u - u_fit
        mad    = np.median(np.abs(resid - np.median(resid)))
        scale  = 1.4826 * mad if mad > 1e-12 else 1.0
        w      = (np.abs(resid) / scale <= c).astype(float)
        w      = np.maximum(w, 1e-6)
    return u_fit


def apply_projection_from_train_df(train_df, pred_residual, deg=5, n_iter=4, c=2.0):
    """
    Apply per-well robust projection using columns already in train_df.

    train_df columns used:
      well        â€“ well identifier (str)
      md_since    â€“ eval_MD - last_known_MD  (relative position, â‰¥ 0)
      z           â€“ Z coordinate of eval row
      dz          â€“ z - last_known_Z

    Derivation:
      anchor       = last_known_tvt + last_known_Z
      U            = pred_abs + Z - anchor
                   = (pred_resid + last_known_tvt) + z - (last_known_tvt + last_known_z)
                   = pred_resid + (z - last_known_z)
                   = pred_resid + dz
      s            = md_since / max(md_since) per well   [0, 1]
      corrected_residual = U_fit - dz
    """
    corrected = pred_residual.copy().astype(np.float32)
    n_done = 0

    well_col = train_df['well'].values
    md_since = train_df['md_since'].values.astype(np.float64)
    dz       = train_df['dz'].values.astype(np.float64)

    for well_id in np.unique(well_col):
        idx = np.where(well_col == well_id)[0]
        nh  = len(idx)
        if nh < deg + 1:
            continue
        span = md_since[idx].max()
        if span <= 0:
            continue
        s = md_since[idx] / span
        u = pred_residual[idx].astype(np.float64) + dz[idx]
        u_fit = _robfit(s, u, deg=deg, n_iter=n_iter, c=c)
        corrected[idx] = (u_fit - dz[idx]).astype(np.float32)
        n_done += 1

    print(f'  Projection applied to {n_done} wells')
    return corrected


def apply_projection_from_meta(meta_df, pred_residual, deg=5, n_iter=4, c=2.0):
    """
    Apply projection from a meta DataFrame with columns:
      well, md_since, z, dz
    Used for test wells (built from raw horizontal well CSVs).
    """
    corrected = pred_residual.copy().astype(np.float32)
    n_done = 0

    well_col = meta_df['well'].values
    md_since = meta_df['md_since'].values.astype(np.float64)
    dz_arr   = meta_df['dz'].values.astype(np.float64)

    for well_id in np.unique(well_col):
        idx = np.where(well_col == well_id)[0]
        nh  = len(idx)
        if nh < deg + 1:
            continue
        span = md_since[idx].max()
        if span <= 0:
            continue
        s = md_since[idx] / span
        u = pred_residual[idx].astype(np.float64) + dz_arr[idx]
        u_fit = _robfit(s, u, deg=deg, n_iter=n_iter, c=c)
        corrected[idx] = (u_fit - dz_arr[idx]).astype(np.float32)
        n_done += 1

    print(f'  Projection applied to {n_done} test wells')
    return corrected


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('Projection functions defined.')

In [ ]:
# â”€â”€ Load train_df (eval-zone rows with all projection columns) â”€â”€â”€â”€
train_df = joblib.load(PREP_DIR / 'train_df.joblib')
print(f'train_df: {train_df.shape}  wells: {train_df["well"].nunique()}')

# Verify required projection columns are present
required = ['well', 'md_since', 'z', 'dz', 'last_known_tvt', 'target']
missing  = [c for c in required if c not in train_df.columns]
if missing:
    raise ValueError(f'train_df missing required columns: {missing}')

if TINY_SAMPLE:
    sample_wells = train_df['well'].unique()[:10]
    train_df = train_df[train_df['well'].isin(sample_wells)].reset_index(drop=True)
    print(f'TINY_SAMPLE: {len(sample_wells)} wells, {len(train_df)} rows')

y_eval   = train_df['target'].values.astype(np.float32)
print(f'Eval rows: {len(train_df)}  Target (residual): mean={y_eval.mean():.2f}  std={y_eval.std():.2f}')

In [ ]:
# â”€â”€ Load prior OOF predictions (residual space) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Priority: EXP-007_final_oof > aw-blend_oof (both in residual space)
N_EVAL = len(train_df)
oof_pred_before = None
prior_name_used = None

for prior_name in ['EXP-007_final', 'aw-blend']:
    oof_path = PRIOR_OOF_DIR / f'{prior_name}_oof.npy'
    if oof_path.exists():
        arr = np.load(oof_path).astype(np.float32)
        if arr.shape[0] == N_EVAL:
            oof_pred_before  = arr
            prior_name_used  = prior_name
            print(f'Loaded OOF from: {prior_name}  shape: {arr.shape}')
            break
        else:
            print(f'Shape mismatch {prior_name}: {arr.shape[0]} vs {N_EVAL} â€” skipping')

if oof_pred_before is None:
    print('WARNING: No prior OOF found â€” using pf_ancc residual as placeholder')
    oof_pred_before = (train_df['pf_ancc'].values - train_df['last_known_tvt'].values).astype(np.float32)
    prior_name_used = 'pf_ancc_placeholder'

# Load test OOF if available
oof_test_before = None
if prior_name_used and prior_name_used not in ('pf_ancc_placeholder',):
    test_path = PRIOR_OOF_DIR / f'{prior_name_used}_test.npy'
    if test_path.exists():
        oof_test_before = np.load(test_path).astype(np.float32)
        print(f'Loaded test OOF: {test_path}  shape: {oof_test_before.shape}')

print(f'OOF RMSE BEFORE projection (residual): {rmse(y_eval, oof_pred_before):.5f}')
print(f'Reference (aw pipeline): 10.45212  (LB 9.964)')

In [ ]:
# â”€â”€ Apply projection â€” train OOF â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
t0 = time.time()
oof_corrected = apply_projection_from_train_df(
    train_df, oof_pred_before,
    deg=PROJ_DEG, n_iter=PROJ_ITERS, c=PROJ_C
)
print(f'  Train projection done in {time.time()-t0:.1f}s')

rmse_before = rmse(y_eval, oof_pred_before)
rmse_after  = rmse(y_eval, oof_corrected)

print('=' * 50)
print(f'OOF RMSE BEFORE: {rmse_before:.5f}')
print(f'OOF RMSE AFTER:  {rmse_after:.5f}')
print(f'Delta:           {rmse_after - rmse_before:+.5f}')
print('=' * 50)

# Per-well breakdown
wb, wa = [], []
for w, idx in pd.DataFrame({'well': train_df['well'].values}).groupby('well').groups.items():
    idx = idx.values
    wb.append(rmse(y_eval[idx], oof_pred_before[idx]))
    wa.append(rmse(y_eval[idx], oof_corrected[idx]))
print(f'Per-well median RMSE â€” before: {np.median(wb):.4f}  after: {np.median(wa):.4f}')
print(f'% wells improved: {100*np.mean(np.array(wa) < np.array(wb)):.1f}%')

In [ ]:
# â”€â”€ Apply projection â€” test OOF â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Build test projection metadata from raw horizontal well CSVs.
# test_meta has columns: well, md_since, z, dz, last_known_tvt
test_corrected = None

if oof_test_before is None:
    print('No test OOF available â€” skipping test projection.')
elif DATA_DIR is None:
    print('DATA_DIR not found â€” cannot build test anchors. Skipping test projection.')
    test_corrected = oof_test_before
else:
    print('Building test projection metadata from raw CSVs...')
    test_hw_paths = sorted((DATA_DIR / 'test').glob('*__horizontal_well.csv'))
    meta_rows = []
    for p in test_hw_paths:
        wid = p.stem.replace('__horizontal_well', '')
        try:
            hw = pd.read_csv(p, usecols=['MD', 'Z', 'TVT_input'])
        except Exception:
            hw = pd.read_csv(p)
        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(kn) == 0 or len(ev) == 0:
            continue
        lk      = kn.iloc[-1]
        last_md = float(lk['MD'])
        last_z  = float(lk['Z'])
        last_tvt = float(lk['TVT_input'])
        for _, r in ev.iterrows():
            meta_rows.append({
                'well':          wid,
                'md_since':      float(r['MD']) - last_md,
                'z':             float(r['Z']),
                'dz':            float(r['Z']) - last_z,
                'last_known_tvt': last_tvt,
            })
    test_meta = pd.DataFrame(meta_rows)

    if TINY_SAMPLE:
        sample_wells_set = set(train_df['well'].unique())
        test_meta = test_meta[test_meta['well'].isin(sample_wells_set)].reset_index(drop=True)

    print(f'test_meta: {test_meta.shape}  oof_test_before: {oof_test_before.shape}')
    if len(test_meta) == len(oof_test_before):
        test_corrected = apply_projection_from_meta(
            test_meta, oof_test_before,
            deg=PROJ_DEG, n_iter=PROJ_ITERS, c=PROJ_C
        )
    else:
        print(f'WARNING: test_meta length {len(test_meta)} != oof_test_before {len(oof_test_before)} â€” skipping')
        test_corrected = oof_test_before

if test_corrected is not None:
    print(f'Test corrected shape: {test_corrected.shape}')

In [ ]:
# â”€â”€ Save â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
np.save(WORK_DIR / f'{EXP_ID}_oof.npy', oof_corrected)
print(f'Saved: {WORK_DIR / f"{EXP_ID}_oof.npy"}  shape: {oof_corrected.shape}')

if test_corrected is not None:
    np.save(WORK_DIR / f'{EXP_ID}_test.npy', test_corrected)
    print(f'Saved: {WORK_DIR / f"{EXP_ID}_test.npy"}  shape: {test_corrected.shape}')

result = {
    'exp_id': EXP_ID,
    'prior': prior_name_used,
    'rmse_before': round(rmse_before, 6),
    'rmse_after':  round(rmse_after,  6),
    'delta': round(rmse_after - rmse_before, 6),
    'pct_wells_improved': round(float(np.mean(np.array(wa) < np.array(wb))), 4),
    'proj_deg': PROJ_DEG, 'proj_iters': PROJ_ITERS, 'proj_c': PROJ_C,
    'tiny_sample': TINY_SAMPLE,
    'io_note': 'OOF in RESIDUAL space (TVT - last_known_tvt).',
}
with open(WORK_DIR / f'{EXP_ID}_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))
print('\nNext: record in experiment_queue.md + memory/previous_runs.md')
print('Then: use rogii-submission.ipynb to produce final submission.csv')